## AI4Climate ML tutorial - Building a machine learning model with gridded data
* Author: Stephen Haddad
* Affiliation: UK Met Office
* History: 1.0
* Last update: 2026-03-16
* © British Crown Copyright 2017-2026, Met Office. Please see LICENSE.md for license details.

## Overview of the broad topic covered

So far we have been working data in tabular format when training machine learning algorithms. We have previously encountered **gridded data**, but previously we converted this into a tabular format before using it with machine learning tools. In this notebook we will instead feed in gridded data into the training and inference of machine learning algorithms to see how working with gridded data is different to working with tabular data.





### Prerequisites 
- Same as previous  notebooks
- Have completed the training pipeline, inference and evaluation notebooks.


### Learning outcomes from completing the notebook

- Understand how to iterate through a gridded dataset for training
- Understand  how to create a convolutional neural network to work with gridded data.
- Understand how to evaluate gridded output for a regression type problem.

## Tutorial - Creating a machine learning pipe

Further Reading
* [PyTorch Docs](https://pytorch.org/)

### Environment 
This notebook uses the environment defined in this repository in the [requirements.txt](../requirements.txt) file (venv) or [requirements.yaml](../requirements.yaml).  

## Tutorial 
a balance of explanation and activity



#### Import libraries
Key libraries for this tutorial include:
- Xarray for loading the input dataset
- scikit learn for preparing the data for training
- pytorch for creating and training the neural network
- matplotlib and cartopy for visualising the results
- scores for evaluating model performance

In [2]:
import pathlib
import os
import datetime
import json
import re

In [3]:
import numpy 
import xarray

In [4]:
import matplotlib
import matplotlib.pyplot
import cartopy.crs

In [5]:
import sklearn
import sklearn.preprocessing
import sklearn.tree


The tutorial config from the JSON file.

In [7]:
with open ('config.json','r') as tutorial_config:
    tutorial_config = json.load(tutorial_config)
tutorial_config

{'platform': 'jasmin',
 'default_dirs': {'mo_linux': '/data/users/dscop/ml_tutorial/',
  'jasmin': '/gws/ssde/j25a/mmh_storage/ai4c_data/'},
 'random_seed': 12345,
 'climate_subgroups': {'non-land': 'None',
  'Af': 'Tropical, rainforest',
  'Am': 'Tropical, monsoon',
  'Aw': 'Tropical, savannah',
  'BWh': 'Arid, desert, hot',
  'BWk': 'Arid, desert, cold',
  'BSh': 'Arid, steppe, hot',
  'BSk': 'Arid, steppe, cold',
  'Csa': 'Temperate, dry summer, hot summer',
  'Csb': 'Temperate, dry summer, warm summer',
  'Csc': 'Temperate, dry summer, cold summer',
  'Cwa': 'Temperate, dry winter, hot summer',
  'Cwb': 'Temperate, dry winter, warm summer',
  'Cwc': 'Temperate, dry winter, cold summer',
  'Cfa': 'Temperate, no dry season, hot summer',
  'Cfb': 'Temperate, no dry season, warm summer',
  'Cfc': 'Temperate, no dry season, cold summer',
  'Dsa': 'Cold, dry summer, hot summer',
  'Dsb': 'Cold, dry summer, warm summer',
  'Dsc': 'Cold, dry summer, cold summer',
  'Dsd': 'Cold, dry summer

In [8]:
def get_platform_dir(select_platform, config):
    try:
        root_path = pathlib.Path(config['default_dirs'][select_platform]) / 'weatherbench'
    except KeyError:
        root_path = pathlib.Path(os.environ['HOME']) / 'weatherbench'
    return root_path

In [9]:
current_platform = tutorial_config['platform']

In [10]:
current_platform

'jasmin'

In [11]:
root_data_dir = get_platform_dir(current_platform, tutorial_config)

print(root_data_dir.is_dir())
root_data_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/weatherbench')

Define the key parameters for this experiment.

In [12]:
resolution_dict = {5.625: '5.625deg'}

In [13]:
weatherbench_dir = root_data_dir / resolution_dict[5.625]
print(weatherbench_dir.is_dir())
weatherbench_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/weatherbench/5.625deg')

In [14]:
var_list = {
    'temperature': [850, 500],
    'geopotential': [500],
}

In [55]:
era5_rename_lut = {
    'z': 'geopotential',
    't': 'temperature',
    'q': 'specific_humidity',
    'u': 'u_component_of_wind',
    'v': 'v_component_of_wind',
}

In [16]:
def get_rename_dict(xr_ds, var_lut):
    vars_present = list(xr_ds.data_vars)
    rename_lut = {k1: v1 for k1,v1 in var_lut.items() if k1 in vars_present}
    return rename_lut
                  

In [56]:
var_list = ['temperature', 
            'specific_humidity',
            'u_component_of_wind',
            'v_component_of_wind',
            'geopotential'
           ]

In [50]:
pl_list = [1000,850,700,500,200]

In [52]:
start_period = datetime.datetime(1980,1,1,0,0)
end_period = datetime.datetime(1982,1,1,0,0)

In [80]:
agg_dims = ['time','lat','lon']

In [102]:
var_ds_list = []
var_stats_dict = {}
for current_var in var_list:
    print(current_var)
    pattern = re.compile(current_var + r"_(\d{4})_5\.625deg\.nc")
    files_to_load = sorted([ f1 for f1 in (weatherbench_dir / current_var).iterdir() if pattern.match(f1.name)])
    current_ds = xarray.open_mfdataset(files_to_load)
    current_ds = current_ds.loc[{'level': pl_list, 
                                 'time': slice(start_period,end_period)}]
    current_ds = current_ds.rename(get_rename_dict(current_ds, era5_rename_lut))
    current_ds.chunk({'time': 240})
    current_std = current_ds.std(dim=agg_dims)
    current_mean = current_ds.mean(dim=agg_dims)
    var_stats_dict[current_var] = {
        'mean': current_mean,
        'std': current_std,
    }
    current_ds = (current_ds - current_mean) / current_std
    
    var_ds_list += [current_ds]


temperature
specific_humidity
u_component_of_wind
v_component_of_wind
geopotential


In [124]:
len(era5_norm_ds['time']) / 5

3509.0

In [126]:
era5_norm_ds = xarray.merge(var_ds_list).chunk({'time':240})
era5_norm_ds

<xarray.Dataset> Size: 4GB
Dimensions:              (lon: 64, lat: 32, level: 5, time: 17545)
Coordinates:
  * lon                  (lon) float64 512B 0.0 5.625 11.25 ... 348.8 354.4
  * lat                  (lat) float64 256B -87.19 -81.56 -75.94 ... 81.56 87.19
  * level                (level) int32 20B 1000 850 700 500 200
  * time                 (time) datetime64[ns] 140kB 1980-01-01 ... 1982-01-01
Data variables:
    temperature          (time, level, lat, lon) float32 719MB dask.array<chunksize=(240, 5, 32, 64), meta=np.ndarray>
    specific_humidity    (time, level, lat, lon) float32 719MB dask.array<chunksize=(240, 5, 32, 64), meta=np.ndarray>
    u_component_of_wind  (time, level, lat, lon) float32 719MB dask.array<chunksize=(240, 5, 32, 64), meta=np.ndarray>
    v_component_of_wind  (time, level, lat, lon) float32 719MB dask.array<chunksize=(240, 5, 32, 64), meta=np.ndarray>
    geopotential         (time, level, lat, lon) float32 719MB dask.array<chunksize=(240, 5, 32, 64), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2020-03-03 19:31:21 GMT by grib_to_netcdf-2.16.0: /opt/ecmw...

In [105]:
var_stats_json = { var1: {
    'mean': var_dict['mean'].to_dict()['data_vars'][var1]['data'],
    'std': var_dict['std'].to_dict()['data_vars'][var1]['data'],
} for var1, var_dict in var_stats_dict.items()}

In [131]:
wb_arco_dir_name = 'wb_arco'

In [134]:
wb_arco_zarr_out_dir = pathlib.Path(root_data_dir / wb_arco_dir_name)
print(wb_arco_zarr_out_dir.is_dir())
wb_arco_zarr_out_dir

True


PosixPath('/gws/ssde/j25a/mmh_storage/ai4c_data/weatherbench/wb_arco')

In [136]:
era5_norm_ds.to_zarr(wb_arco_zarr_out_dir)

/gws/nopw/j04/mohc_shared/users/shaddad/venv/ai4c_nb_gpu/lib/python3.12/site-packages/zarr/api/asynchronous.py:229: UserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [137]:
with open(wb_arco_zarr_out_dir / 'stats.json','w') as stats_json_file:
    json.dump(var_stats_json, stats_json_file)

In [155]:
[v1 for v1 in xarray.open_zarr(wb_arco_zarr_out_dir).loc[{'time':'1981-1-2'}].data_vars][0]

['specific_humidity',
 'geopotential',
 'v_component_of_wind',
 'u_component_of_wind',
 'temperature']

In [176]:
ds1 = xarray.open_zarr(wb_arco_zarr_out_dir)['time']

In [177]:
ds1

<xarray.DataArray 'time' (time: 17545)> Size: 140kB
array(['1980-01-01T00:00:00.000000000', '1980-01-01T01:00:00.000000000',
       '1980-01-01T02:00:00.000000000', ..., '1981-12-31T22:00:00.000000000',
       '1981-12-31T23:00:00.000000000', '1982-01-01T00:00:00.000000000'],
      shape=(17545,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 140kB 1980-01-01 ... 1982-01-01
Attributes:
    long_name:  time

In [174]:
numpy.stack(
    [ds1[v1].to_numpy() for v1 in ds1.data_vars],
    axis=1).reshape(-1,len(ds1.data_vars)*len(ds1['level']),32,64).shape


(24, 25, 32, 64)

In [165]:
(len(xarray.open_zarr(wb_arco_zarr_out_dir).loc[{'time':'1981-1-2'}].data_vars) *
len(xarray.open_zarr(wb_arco_zarr_out_dir).loc[{'time':'1981-1-2'}]['level']) )

25

In [175]:

ds1['time']

<xarray.DataArray 'time' (time: 24)> Size: 192B
array(['1981-01-02T00:00:00.000000000', '1981-01-02T01:00:00.000000000',
       '1981-01-02T02:00:00.000000000', '1981-01-02T03:00:00.000000000',
       '1981-01-02T04:00:00.000000000', '1981-01-02T05:00:00.000000000',
       '1981-01-02T06:00:00.000000000', '1981-01-02T07:00:00.000000000',
       '1981-01-02T08:00:00.000000000', '1981-01-02T09:00:00.000000000',
       '1981-01-02T10:00:00.000000000', '1981-01-02T11:00:00.000000000',
       '1981-01-02T12:00:00.000000000', '1981-01-02T13:00:00.000000000',
       '1981-01-02T14:00:00.000000000', '1981-01-02T15:00:00.000000000',
       '1981-01-02T16:00:00.000000000', '1981-01-02T17:00:00.000000000',
       '1981-01-02T18:00:00.000000000', '1981-01-02T19:00:00.000000000',
       '1981-01-02T20:00:00.000000000', '1981-01-02T21:00:00.000000000',
       '1981-01-02T22:00:00.000000000', '1981-01-02T23:00:00.000000000'],
      dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 192B 1981-01-02 ... 1981-01-02T23:00:00
Attributes:
    long_name:  time